# Shami Fine-Tuning

## Dowload Libraries

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade trl datasets transformers
print('done!')

## Check if GPU Exists

In [32]:
import torch

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

supports_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
dtype = torch.bfloat16 if supports_bf16 else torch.float16
print(f'dtype: {dtype}')

GPU: NVIDIA L4
VRAM: 23.7 GB
dtype: torch.bfloat16


## Download Model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen3.5-2B',
    max_seq_length=512,
    dtype=dtype,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

model.print_trainable_parameters()

## Dataset

In [34]:
import json
from datasets import Dataset

JSONL_FILE = 'data/shami_dataset.jsonl'

records = []
with open(JSONL_FILE, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'total lines: {len(records)}')
print(json.dumps(records[0], ensure_ascii=False, indent=2))

total lines: 2234
{
  "instruction": "اشرحلي شو المجاز اللي استعملناه بهي الجملة.",
  "input": "قلبو صخر وما بيلين.",
  "output": "لما بنقول عن حدا \"قلبو حجر\"، يعني هاد الزلمة ميت قلبو وما بيحس، لا عندو رحمة ولا بيعرف شو يعني شفقة. هي الكلمة بتشبه قلبو بالصخرة الصوان، قاسية وما بتلين، وبتبينلك إنو هاد الشخص صعب كتير الواحد يوصل لمشاعرو أو يأثر فيه."
}


## Change Data Format to ChatML

In [35]:
SYSTEM_MSG = 'أنت مساعد ذكي بتحكي باللهجة الشامية السورية.'

def format_example(example):
    instruction = example.get('instruction', '')
    inp = example.get('input', '')
    output = example.get('output', '')
    user_msg = f'{instruction}\n\n{inp}'.strip() if inp else instruction
    text = (
        f'<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n'
        f'<|im_start|>user\n{user_msg}<|im_end|>\n'
        f'<|im_start|>assistant\n{output}<|im_end|>'
    )
    return {'text': text}

dataset = Dataset.from_list(records).map(format_example)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')
print('\Example:')
print(train_dataset[0]['text'][:300])

Map:   0%|          | 0/2234 [00:00<?, ? examples/s]

Train: 2010 | Eval: 224
\Example:
<|im_start|>system
أنت مساعد ذكي بتحكي باللهجة الشامية السورية.<|im_end|>
<|im_start|>user
وصفلي موقف لما كان لازم تظهر فيه قوة عقلك وصلابة تفكيرك.<|im_end|>
<|im_start|>assistant
كان لازم أحكي بقوة وأكون ذكي بموقف صعب كتير. كنت طالب جديد بالجامعة، وكنت عم بحاول لاقي حالي وأحس إني منتمي لهالمكان. كن


## Training

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./model/shami-qwen-output',
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-3,
    lr_scheduler_type='cosine',
    fp16=(dtype == torch.float16),
    bf16=(dtype == torch.bfloat16),
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=50,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    dataset_text_field='text',
    max_seq_length=512,
    optim='adamw_8bit',
    seed=3407,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print('Start training')
trainer.train()
print('Training is done')

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/2010 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/224 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


بدأ التدريب...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,010 | Num Epochs = 4 | Total steps = 504
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 10,911,744 of 2,224,153,408 (0.49% trained)


Step,Training Loss,Validation Loss
50,2.562716,2.445612
100,2.614921,2.470264
150,2.265666,2.458418
200,2.212494,2.401481
250,2.256384,2.288659
300,1.653288,2.307732
350,1.637543,2.204757
400,1.000774,2.298602
450,0.927935,2.303423
500,0.908861,2.294919


Unsloth: Not an error, but Qwen3_5ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


خلص التدريب!


## Save Model

In [ ]:
SAVE_PATH = './model/shami-qwen-final'

# Save Only LoRA adapeters
trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'LoRA adapters saved in: {SAVE_PATH}')

# Save LoRA adapeters With Original Model
# FastLanguageModel.save_pretrained_merged(
#     model=trainer.model,
#     tokenizer=tokenizer,
#     save_directory='./shami-qwen-merged',
# )
# print('Total model saved in: shami-qwen-merged')

## Try Model

In [31]:
from transformers import AutoTokenizer as HFTokenizer

# استخدم AutoTokenizer مباشرة لتجنب مشاكل multimodal processor
hf_tok = HFTokenizer.from_pretrained('unsloth/Qwen3.5-2B')

SYSTEM_MSG = ''

def chat(prompt: str, max_new_tokens: int = 200) -> str:
    text = (
        f'<|im_start|>system\n{SYSTEM_MSG}<|im_end|>\n'
        f'<|im_start|>user\n{prompt}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = hf_tok(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            do_sample=True,
            repetition_penalty=1.3,
            pad_token_id=hf_tok.eos_token_id,
        )
    response = hf_tok.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response


questions = 'كيف فيني استخدم الذكاء الاصطناعي بالمجال الصحي؟'


print(f'السؤال: {questions}')
print(f'الجواب: {chat(questions)}')

السؤال: كيف فيني استخدم الذكاء الاصطناعي بالمجال الصحي؟
الجواب: من أشهر استخدامات الـ AI بالصحة هي نظام الرعاية الصحية. بيتستخدم هالنظام لنعمل خطط علاجية مخصصة لكل مرض أو حالة، وهيك بتساعد الأطباء يحددوا الأنماط اللي ممكنين يصير فيها المرض ويوصيوك بالخطط العلاجية المناسبة لحالها. وكمان بيساعدو ينشروا المعلومات الطبية من أي مكان وبيسهل على المرضى يتعاملوا مع أمراضهم بأقل تعقيد وأكتر مرونة لما بدخول المريض عالبدايو الطبي. وفوق هاد كلهه، كمان بيخلي رعاية السريل عندهن مصاري أقل ومنقدر يشتغلوا بدون ما يحتاجوا طبيبًا كتار للزيارات اليومية.
----------------------------------------


## Test Original Model

In [14]:
from unsloth import FastLanguageModel
import torch

# Load the base model without PEFT adapters
# dtype and hf_tok are already defined in previous cells
base_model, _ = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen3.5-2B',
    max_seq_length=512,
    dtype=dtype, # use the same dtype as before
    load_in_4bit=True,
)
print('Original model loaded!')

==((====))==  Unsloth 2026.3.3: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Original model loaded!


In [20]:
# Define a chat function for the original model
# Reusing the SYSTEM_MSG from the fine-tuning setup for consistency
SYSTEM_MSG_FOR_ORIGINAL_INFERENCE = ''

def chat_original_model(prompt: str, max_new_tokens: int = 256) -> str:
    text = (
        f'<|im_start|>system\n{SYSTEM_MSG_FOR_ORIGINAL_INFERENCE}<|im_end|>\n'
        f'<|im_start|>user\n{prompt}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    # hf_tok is already loaded from 'unsloth/Qwen3.5-2B' in cell Zwr6TRIUhdga
    inputs = hf_tok(text, return_tensors='pt').to(base_model.device)
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
            do_sample=True,
            repetition_penalty=1.3,
            pad_token_id=hf_tok.eos_token_id,
        )
    response = hf_tok.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    # Remove <think> and </think> tags
    response = response.replace('<think>', '').replace('</think>', '').strip()
    return response
print('Original model inference function defined!')

Original model inference function defined!


In [30]:
question_for_original = 'كيف فيني استخدم الذكاء الاصطناعي بالمجال الصحي؟'
print(f'السؤال: {question_for_original}')
print(f'الجواب من الموديل الأصلي: {chat_original_model(question_for_original)}')
print('-' * 40)

السؤال: كيف فيني استخدم الذكاء الاصطناعي بالمجال الصحي؟
الجواب من الموديل الأصلي: يستخدم الأطباء والمختصون بالذكاء الاصطناعي (AI) بشكل متزايد والمستمر لتحسين دقة التشخيص، وزيادة الكفاءة التشغيلية، وتقليل الأخطاء الطبية. إليك أبرز المجالات التي يتم فيها تطبيق هذه التقنيات:

### 1. تشخيص الأمراض والصور السريرية
*   **الاعتماد على الصور:** يستخدم الممرضين والأطباء خوارزميات ذكية لتقييم صور الأشعة المقطعية والتصوير العظمى والسكروماتوغرام بدقة عالية جداً من خلال تحليل الأنماط غير المرئية للعين البشرية بسرعة أكبر. هذا يساعد في اكتشاف التغيرات الطفيفة قبل ظهورها بوضوح.
*   **التشخيص المبكر:** يمكن للنظام العصبي أو البصري أن يسجل التاريخ المرضي تلقائياً ويحلله مع البيانات المختبرية لتقديم توصيف طبي دقيق وسريعاً للمريض الذي لا يستطيع التعبير عن نفسه بسبب الحزن الشديد أو الألم المزمن.

### 2. إدارة السجلات الصحية الإلكترونية (EHRs) وخدمات الدعم الطبي
تقوم أنظمة مثل "MediQ" بتقسيم المهام الإدارية المعقدة إلى مهام بسيطة ومقابلة للحل الآلي عبر واجهة مستخدم
----------------------------------------
